# DETECCIÓN DE INEFICIENCIAS EN PLANTA FOTOVOLTAICA - PARTE 2

En esta segunda fase analizamos los datos preparados en el notebook anterior con el objetivo de identificar patrones de comportamiento, evaluar el rendimiento de las instalaciones y detectar posibles ineficiencias operativas en las plantas fotovoltaicas.

## Carga de librerías

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

%config IPCompleter.greedy = True

pd.options.display.float_format = '{:15.2f}'.format
sns.set_style('darkgrid')

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

## Importación de datos

In [ ]:
df = pd.read_pickle('Datos/df.pickle')

In [ ]:
df.head()

In [ ]:
df.shape

### Selección de sensores representativos

Para comparar ambas plantas necesitamos seleccionar sensores que representen bien sus condiciones. Vamos a usar los sensores ambientales (irradiación, temperatura) de inversores específicos de cada planta.

In [ ]:
recepcion = df.loc[(df.inverter_id == '1BY6WEcLGh8j5v7')  | (df.inverter_id == 'q49J1IKaHRwDQnt'), 'planta':'t_modulo']
recepcion

Listo, ya tenemos un dataset compacto con las variables que nos interesan.

## Análisis de las condiciones operativas

### Comparamos las condiciones ambientales entre plantas

¿Reciben la misma irradiación? ¿Tienen temperaturas similares? Esto nos sirve para saber si las diferencias en producción se deben a condiciones distintas o a problemas técnicos.

In [ ]:
recepcion.groupby('planta')[['irradiacion','t_ambiente','t_modulo']].mean()

Dado que la irradiación representa una magnitud acumulativa, utilizamos su suma para comparar la energía solar recibida por cada planta durante el periodo analizado.

In [ ]:
temp = recepcion.groupby('planta').agg({'irradiacion':sum,'t_ambiente':np.mean,'t_modulo':np.mean})
temp

Observamos diferencias moderadas entre ambas plantas en las variables analizadas. Para facilitar la comparación visual, representamos los resultados mediante gráficos de barras.

In [ ]:
f, ax = plt.subplots(nrows = 1, ncols = 3, figsize = (18,5))

ax[0].bar(temp.index, temp.irradiacion, color = ['red','blue'], alpha = 0.3)
ax[1].bar(temp.index, temp.t_ambiente, color = ['red','blue'], alpha = 0.3)
ax[2].bar(temp.index, temp.t_modulo, color = ['red','blue'], alpha = 0.3);

ax[0].set_title('Irradiacion por planta')
ax[1].set_title('Temperatura ambiente por planta')
ax[2].set_title('Temperatura módulo por planta');

Resumen de comparaciones:

• La planta 2 registra valores ligeramente superiores de irradiación y temperatura

• Las diferencias son pequeñas, así que probablemente no expliquen las grandes diferencias en generación que veremos después

### Relación entre irradiación y temperatura

Vamos a ver cómo se relacionan las variables ambientales entre sí (irradiación con temperatura del módulo, etc.):

In [ ]:
temp = recepcion.loc[:,['planta','irradiacion','t_ambiente','t_modulo']]
temp

In [ ]:
temp.select_dtypes(include='number').corr()

Observamos una relación más intensa entre irradiación y temperatura del módulo que entre irradiación y temperatura ambiente. Para facilitar la interpretación de estos resultados, representamos gráficamente la matriz de correlación.

In [ ]:
sns.heatmap(temp.select_dtypes(include='number').corr(), annot= True);

La correlación es muy alta (0.95) entre irradiación y temperatura del módulo. Esto tiene sentido: a más radiación, más calienta el módulo.

In [ ]:
sns.pairplot(temp, hue ='planta', height = 3, plot_kws = {'alpha': 0.1})

Hay una correlación positiva clara entre irradiación, temperatura ambiente y temperatura del módulo.

La más fuerte es entre irradiación y temperatura del módulo, lo que tiene sentido físico.

### Patrones horarios de irradiación

¿Cómo se distribuye la irradiación a lo largo del día en cada planta? ¿A qué hora genera más? Vamos a hacer una tabla cruzada (hora vs planta):

In [ ]:
# Guardamos la tabla como objeto temporal:
temp = pd.crosstab(recepcion.hora, recepcion.planta, values = recepcion.irradiacion, aggfunc = 'mean')
temp

Obtenemos la tabla cruzada, donde cada columna representa una planta y cada fila una hora del día.

Observamos ausencia de irradiación durante la noche y máximos en torno a las 11 y 12 horas.

A partir de la tarde la irradiación disminuye progresivamente hasta desaparecer.

Representamos esta información mediante un mapa de calor para facilitar su interpretación visual.

In [ ]:
plt.figure(figsize = (10,10))
sns.heatmap(temp,annot = True, fmt = '.2f');

Patrón similar en ambas plantas: máximos entre las 11:00 y 13:00 horas. La planta 2 registra valores ligeramente superiores durante el mediodía. Esto sugiere que ambas plantas reciben irradiación comparable.

Veamos la temperatura ambiente para completar la comparación:

Mismo análisis con temperatura ambiente:

In [ ]:
temp = pd.crosstab(recepcion.hora, recepcion.planta, values = recepcion.t_ambiente, aggfunc = 'mean')
temp

In [ ]:
plt.figure(figsize = (10,10))
sns.heatmap(temp,annot = True, fmt = '.2f');

Observamos un desfase temporal significativo entre el máximo de irradiación y el máximo de temperatura ambiente, con un retraso aproximado de 2 a 3 horas. Este retraso se atribuye a la inercia térmica característica de los procesos de calentamiento atmosférico.

**Conclusiones principales:**

- Ambas plantas presentan patrones de comportamiento térmico similares, lo que sugiere condiciones ambientales comparables.
- La planta 2 registra temperaturas ligeramente más elevadas durante la mayor parte del día.
- La proximidad geográfica de ambas instalaciones se ve reflejada en la similitud de los patrones observados.
- El período con irradiación significativa y capacidad potencial de generación de DC se extiende aproximadamente entre las 7:00 y las 17:00 horas.
- El pico máximo de irradiación ocurre alrededor de las 12:00 horas.
- Las temperaturas máximas ambiente se registran entre las 14:00 y las 16:00 horas, evidenciando el desfase temporal mencionado.
- Este desfase temporal contribuye a explicar la correlación relativamente menor observada entre irradiación y temperatura ambiente en comparación con la correlación entre irradiación y temperatura del módulo.

## Problema principal: ¿Por qué genera tanto menos DC la planta 2?

Primer hallazgo raro: teniendo prácticamente la misma irradiación, la planta 2 genera mucho menos DC que la planta 1. Vamos a investigar qué pasa.

In [ ]:
plt.figure(figsize = (12,8))
sns.scatterplot(data = df, x = df.irradiacion, y = df.kw_dc);

Se ven dos grupos diferenciados en el gráfico. Para ver a qué planta corresponde cada grupo, añado el color por planta:

In [ ]:
plt.figure(figsize = (12,8))
sns.scatterplot(data = df, x = df.irradiacion, y = df.kw_dc, hue = 'planta');

El análisis por planta revela un hallazgo significativo: para niveles de irradiación equivalentes, la planta 2 produce considerablemente menos potencia DC en comparación con la planta 1. Este comportamiento diferenciado sugiere ineficiencias operativas que requieren investigación adicional.

De manera adicional, se ha observado que la planta 1 presenta una relación anómala entre la potencia DC generada y la potencia AC inyectada a la red, sugiriendo posibles problemas en el proceso de inversión.

Para profundizar en el análisis de estas anomalías, se explora la relación entre irradiación y la variable acumulada de energía generada:

In [ ]:
plt.figure(figsize = (12,8))
sns.scatterplot(data = df, x = df.irradiacion, y = df.kw_dia, hue = 'planta');

El gráfico resultante presenta una dispersión significativa que requiere análisis adicional. Se puede ver que la relación esperada entre irradiación e energía acumulada presenta un patrón no lineal, lo que a priori podría interpretarse como anómalo.

Sin embargo, hay que considerar que `kw_dia` es una variable acumulativa. A lo largo del día acumula energía continuamente, sin importar si en ese momento hay irradiación máxima o no. Por eso los valores máximos de energía acumulada se alcanzan al final del día, cuando no hay irradiación pero ya se ha acumulado toda la del día.

Para verificarlo y evitar malinterpretaciones, vamos a ver cómo evoluciona el acumulado según la hora:

In [ ]:
df.groupby('hora')[['kw_dia']].mean().plot.bar();

El acumulado sube a lo largo del día, como es lógico. Pero hay saltos raros entre las 18:00-23:00 y entre las 5:00-6:00 que parecen ser problemas en los datos.

De todas formas, el análisis principal es otro: entender las diferencias de generación entre plantas, no estos detalles de calidad de datos.

### Diferencia importante en generación de DC

Para el mismo nivel de irradiación, la planta 2 genera menos potencia DC que la planta 1. Esta diferencia es consistente y sugiere problemas en los sistemas de generación de la planta 2.

In [ ]:
### KPIs Principales - Comparación por Planta

# Período: horas de luz (8:00-17:00) con datos válidos
temp = df.between_time('08:00:00','17:00:00').copy()
temp = temp[temp['irradiacion'] > 0]

kpis = temp.groupby('planta').agg({
    'irradiacion': 'mean',
    'kw_dc': 'mean',
    'kw_ac': 'mean',
}).round(2)

# Eficiencia media
temp['eficiencia'] = np.where(temp['kw_dc'] > 0, (temp['kw_ac'] / temp['kw_dc']) * 100, 0)
kpis['eficiencia_dc_ac_%'] = temp.groupby('planta')['eficiencia'].mean().round(2)

print("KPI SUMMARY BY PLANT")
print("=" * 70)
print(f"{'Metric':<30} {'Plant 1':<20} {'Plant 2':<20}")
print("=" * 70)
print(f"{'Avg Irradiation (W/m²)':<30} {kpis.loc['p1','irradiacion']:<20} {kpis.loc['p2','irradiacion']:<20}")
print(f"{'Avg DC Power (kW)':<30} {kpis.loc['p1','kw_dc']:<20} {kpis.loc['p2','kw_dc']:<20}")
print(f"{'Avg AC Power (kW)':<30} {kpis.loc['p1','kw_ac']:<20} {kpis.loc['p2','kw_ac']:<20}")
print(f"{'Avg DC→AC Efficiency (%)':<30} {kpis.loc['p1','eficiencia_dc_ac_%']:<20} {kpis.loc['p2','eficiencia_dc_ac_%']:<20}")
print("=" * 70)

Vamos a asumir que los datos de DC son correctos (aunque con las salvedades anteriores) y ver cómo evoluciona la generación a lo largo de los días. Si hay algún patrón o anomalía se debería ver aquí.

In [ ]:
df_dia = pd.read_pickle('Datos/df_dia.pickle')
df_dia.head()

In [ ]:
# Vamos a representarlo en seaborn con la evolucion de DC en una línea para cada planta
plt.figure(figsize = (10,8))
sns.lineplot(data = df_dia, x= df_dia.index, y = 'kw_dc_sum', hue = 'planta');

El análisis visual de la serie temporal de generación diaria revela una diferencia sustancial en los perfiles de producción de potencia DC entre ambas plantas.

La planta 2 presenta un patrón de generación más simétrico y regular a lo largo del período observado, mientras que la planta 1 exhibe fluctuaciones más pronunciadas con varios picos irregulares. Esta disparidad en los patrones operacionales sugiere que los problemas identificados previamente en la planta 1 podrían tener una naturaleza sistemática o recurrente.

### Análisis diario con múltiples subgráficos

Para ver con detalle qué pasó cada día, voy a usar Pandas para generar múltiples gráficos automáticamente. La idea es:
1. Extraer la fecha del índice
2. Agrupar por día
3. Dejar que Pandas haga un gráfico para cada día

De esta forma podemos ver anomalías día a día

In [ ]:
df['date'] = df.index.date
df

Ya tenemos la columna de fechas. Ahora agrupamos por plantas para cada una por separado (así es más limpio).

### Analizamos planta 1 día a día

In [ ]:
df[df.planta == 'p1']

In [ ]:
df[df.planta == 'p1'].groupby(['planta','date','hora']).kw_dc.sum()

In [ ]:
# Ha salido bien y lo guardamos en un objeto:
dc_constante_p1 = df[df.planta == 'p1'].groupby(['planta','date','hora']).kw_dc.sum()
dc_constante_p1

Necesito convertir las fechas a columnas para que Pandas genere múltiples subgráficos:

In [ ]:
dc_constante_p1.unstack(level = 1)

De esta forma, cada línea en el gráfico representará un día diferente

In [ ]:
dc_constante_p1.unstack(level = 1).plot(subplots = True, layout = (17,2), sharex = True, figsize = (20,30));

La visualización de múltiples series temporales permite identificar patrones y anomalías en la generación de potencia DC de la planta 1 a lo largo del período analizado.

Se observan fluctuaciones irregulares en varios días del análisis, siendo particularmente notoria la degradación en la generación esperada para el día 19, fecha que coincidiría con el máximo potencial de generación según los patrones de irradiación. De igual forma, se detecta una caída significativa (prácticamente nula) en el día 22, comportamiento que merecería investigación adicional para determinar causas potenciales (paradas de mantenimiento, incidencias técnicas, etc.).

### Hacemos lo mismo con la planta 2

In [ ]:
dc_constante_p2 = df[df.planta == 'p2'].groupby(['planta','date','hora']).kw_dc.sum()
dc_constante_p2

Convertimos las fechas a columnas igual que antes:

In [ ]:
dc_constante_p2.unstack(level = 1).plot(subplots = True, layout = (17,2), sharex = True, figsize = (20,30));

### **Insight 2:** Patrón consistente de baja generación DC en planta 2

La serie temporal de planta 2 es más predecible que la de planta 1. Hay puntos raros pero en general el comportamiento es más regular.

Lo relevante es que la planta 2 genera sistemáticamente menos DC que la planta 1 bajo las mismas condiciones de irradiación. Esto apunta a un problema con los componentes de generación (módulos o conexiones).

## Ahora: ¿Qué pasa en la transformación de DC a AC?

Sabemos que la planta 2 genera menos DC. Pero cuando sí genera DC, ¿la convierte bien a AC? Y ¿por qué la planta 1 genera tanto DC pero convierte tan poco a AC?

Vamos a mirar la relación DC entrada vs AC salida en ambas plantas:

In [ ]:
sns.scatterplot(data = df, x = df.kw_dc, y = df.kw_ac, hue = df.planta);

El análisis del diagrama de dispersión revela un patrón diferenciado entre plantas: para niveles equivalentes de potencia DC de entrada, la planta 2 logra producir significativamente más potencia AC que la planta 1. Este hallazgo apunta a un problema de eficiencia en los inversores de la planta 1, mientras que los equipos de inversión de la planta 2 operan dentro de parámetros de rendimiento esperados.

Para profundizar en la identificación de las causas raíz de la baja eficiencia en la planta 1, se analiza la eficiencia de inversión como función de la hora del día. Este análisis permitirá determinar si el problema es constante a lo largo del día o presenta variaciones horarias que sugieran causas específicas (sobrecalentamiento, pérdidas resistivas, etc.).

In [ ]:
temp = df.groupby(['planta','hora'], as_index = False).eficiencia.mean()
temp

In [ ]:
sns.lineplot(data = temp, x ='hora', y = 'eficiencia', hue = 'planta');

### **Insight 3:** Baja eficiencia de inversión en planta 1

La eficiencia es solo el 10%, cuando debería estar entre el 90-95% en una planta normal. Claramente hay un problema.

Dos preguntas:
- ¿Es un problema generalizado o solo en algunos inversores?
- ¿Por qué la planta 2 pierde eficiencia en mediodía?

Investigo la segunda primero:

In [ ]:
### KPIs de Eficiencia por Inversor

# Inversores con mejores y peores eficiencias
temp = df.between_time('08:00:00','17:00:00').copy()
temp = temp[(temp['kw_dc'] > 0) & (temp['irradiacion'] > 0)]
temp['eficiencia'] = (temp['kw_ac'] / temp['kw_dc']) * 100

# Resumen por inversor
inverter_summary = temp.groupby(['planta', 'inverter_id']).agg({
    'eficiencia': 'mean',
    'kw_dc': 'count'
}).rename(columns={'kw_dc': 'horas_activas'})

inverter_summary = inverter_summary[inverter_summary['horas_activas'] > 10].sort_values('eficiencia')

print("\nINVERTER EFFICIENCY ANALYSIS")
print("=" * 60)
print(f"{'Plant':<10} {'Inverter':<15} {'Avg Eff (%)':<15} {'Active Hours':<20}")
print("=" * 60)
for (planta, inv_id), row in inverter_summary.iterrows():
    print(f"{planta:<10} {inv_id:<15} {row['eficiencia']:<15.1f} {row['horas_activas']:<20.0f}")
print("=" * 60)

In [ ]:
temp = df[['planta','hora','kw_dc','kw_ac']].melt(id_vars= ['planta','hora'])
temp

In [ ]:
plt.figure(figsize = (12,8))
sns.lineplot(data = temp[temp.planta == 'p2'], x= 'hora', y = 'value', hue= 'variable', errorbar= ('ci',False));

Se ve que en las horas centrales hay pérdida de eficiencia en planta 2, pero no es tan grave como parecía por el análisis previo. Hay algo raro que genera ese comportamiento. Vamos a analizar la distribución de eficiencia en esas horas centrales:

In [ ]:
temp = df.between_time('08:00:00','15:00:00')
temp = temp[temp.planta == 'p2']

In [ ]:
temp.eficiencia.plot.density();

Se observa un pico en eficiencia cero que afecta significativamente a los datos. Hay que entender por qué: ¿es un problema de los inversores o simplemente hay momentos sin generación de DC?

In [ ]:
temp[temp.kw_dc ==0]

Parece que el culpable son momentos sin generación de DC, no un fallo del inversor. Así que la pregunta cambia: ¿Por qué a veces no hay DC en la planta 2 durante el mediodía? Filtramos solo momentos con DC > 0:

In [ ]:
temp[temp.kw_dc > 0].eficiencia.plot.density();

Cuando hay DC (kw_dc > 0), la eficiencia es >96%. Así que la pregunta es: ¿por qué a veces no hay DC? ¿Hay un patrón?

In [ ]:
temp['kw_dc_cero'] = np.where(temp['kw_dc'] == 0,1, 0)
temp

In [ ]:
temp.groupby('kw_dc_cero')[['irradiacion','t_ambiente','t_modulo']].mean()

En la temperatura ambiente no hay gran diferencia, pero en la del módulo sí. Aparentemente, cuando no hay DC los módulos están más fríos. Pero, ¿podría ser que si se calienta demasiado el módulo deje de generar DC? Vamos a verificarlo comparando temperatura del módulo con generación de DC:

In [ ]:
Veamos la relación: ¿los DC cero están relacionados con temperaturas bajas o altas?

No se confirma la teoría anterior. Vemos generación de DC incluso a temperaturas altas, y hay DC cero en prácticamente todos los rangos de temperatura. Entonces la temperatura del módulo no es el culpable.

Cambio de enfoque: ¿y si el problema está en los inversores específicos? Vamos a ver qué inversor tiene los mayores problemas de generación cero:

In [ ]:
temp.groupby('inverter_id').kw_dc_cero.mean().sort_values(ascending = False).plot.bar();

Hay diferencias significativas entre inversores: algunos tienen <5% de horas sin DC, otros >30%. 

### **Insight 4**: 

En la planta 2, algunos inversores tienen más momentos sin generación de DC que otros. Esto podría indicar problemas con los módulos conectados a esos inversores.

Veamos si estos inversores al menos funcionan bien cuando sí reciben DC:

In [ ]:
temp[temp.kw_dc>0].groupby(['inverter_id','date'],as_index =False).eficiencia.mean().boxplot(column ='eficiencia',by= 'inverter_id',figsize= (14,10))
plt.xticks(rotation= 90);

**Insight #5**: Cuando hay DC en planta 2, los inversores funcionan bien (>96% eficiencia). El problema no es que los inversores estén rotos, sino que no siempre llega DC.

Veamos cómo se comportan individualmente a lo largo de los días:

In [ ]:
temp[temp.kw_dc>0].groupby(['inverter_id','date']).eficiencia.mean().unstack(level=0).plot(subplots=True, sharex=True, figsize=(20,40))
plt.xticks(rotation= 90);

En estos gráficos se ve que varios inversores tienen comportamientos irregulares (faltan líneas en algunos días). Eso sugiere que tienen problemas intermitentes.

Ahora veamos qué pasa en la planta 1 para comparar:

In [ ]:
temp = df.between_time('08:00:00','15:00:00')
temp= temp[temp.planta =='p1']
temp['kw_de_cero'] =np.where(temp['kw_dc'] ==0,1,0)
temp

In [ ]:
temp.eficiencia.plot.density();

En planta 1 es diferente: todos los inversores tienen la misma eficiencia baja y es constante en todos. Parece que el problema es generalizado, no solo en uno o dos inversores.

In [ ]:
temp.groupby(['inverter_id','date'],as_index =False).eficiencia.mean().boxplot(column='eficiencia', by='inverter_id',figsize=(14,10))
plt.xticks(rotation=90);

La línea es muy plana para todos los inversores: la baja eficiencia es constante en todos ellos y todos los días. Esto refuerza la idea de que es un problema estructural, no puntual.

In [ ]:
temp.groupby(['inverter_id','date']).eficiencia.mean().unstack(level=0).plot(subplots= True, sharex=True, figsize=(20,40))
plt.xticks(rotation=90);

Al menos podemos descartar que sea falta de DC. Todos los inversores de planta 1 reciben DC consistentemente. Así que el problema está definitivamente en la conversión DC-AC.

In [ ]:
temp.groupby('inverter_id').kw_de_cero.mean().sort_values(ascending=False).plot.bar();

En la planta 1 apenas hay momentos sin DC (menos del 2%), así que ese no es el problema. El fallo está en la transformación DC-AC, que es estructural.

### Conclusiones principales

Del análisis realizado se desprenden los siguientes hallazgos:

**Problemas de calidad de datos:**
- Hay inconsistencias en los datos que merecen ser revisadas (especialmente en las horas nocturnas)
- Los medidores de ambas plantas deberían ser verificados

**Generación de potencia DC:**
- **Planta 1**: Genera 10 veces más DC que la planta 2 para el mismo nivel de irradiación. Con una eficiencia de inversión de solo el 10%, esto podría indicar datos escalados incorrectamente.
- **Planta 2**: Genera menos DC pero de manera más consistente. Sin embargo, algunos módulos tienen problemas recurrentes (30%+ de horas sin DC).

**Condiciones ambientales:**
- Ambas plantas reciben irradiación comparable
- La planta 2 registra temperaturas ligeramente superiores, pero esto no explica la diferencia en producción
- El ciclo térmico es el esperado (máxima radiación al mediodía, máxima temperatura por la tarde)

**Inversión DC-AC:**
- **Planta 1**: Muy baja eficiencia (~10%) de manera constante. No es un problema puntual sino estructural. Todos los inversores se ven afectados.
- **Planta 2**: Buena eficiencia (>97%) cuando hay DC de entrada. El problema es que a veces no llega DC, no que falle el inversor.

**Resumen:**
- Planta 1: Problema en la transformación DC → AC (inversores)
- Planta 2: Problema en la generación DC (módulos en ciertos inversores)

## Recomendaciones

1. **Calidad de datos**: Hay anomalías en los datos, especialmente en horas nocturnas. Hay que revisar los medidores para descartar errores de medición.

2. **Planta 2**: Los inversores con >30% de horas sin DC merecen investigación. Podrían haber problemas en los módulos asociados o conexiones deficientes.

3. **Planta 1**: Una eficiencia de inversión de solo el 10% es preocupante. Podría ser un fallo en los inversores o un problema en los datos de DC (posible escalado incorrecto).

---

## Resumen Ejecutivo

### Hallazgos Principales

1. **Condiciones ambientales similares**: Ambas plantas reciben niveles de irradiación prácticamente idénticos (~420 W/m²). No hay diferencias en las condiciones externas.

2. **Diferencia crítica en generación DC**: Para igual irradiación, la planta 2 genera aproximadamente 30-40% menos potencia DC que la planta 1. Esto sugiere problemas en los módulos fotovoltaicos o sus conexiones.

3. **Planta 1: Problema en DC→AC**: A pesar de generar buen DC, la planta 1 convierte apenas el 10-15% a AC. Los inversores no están siendo eficientes en la transformación.

4. **Planta 2: Problema en DC**: Algunos inversores tienen >30% de horas sin generación de DC, lo que indica fallos intermitentes en los módulos asociados.

5. **Impacto global**: Ambas plantas tienen problemas operativos, pero de naturaleza diferente. La planta 2 pierde generación en la primera etapa (DC), mientras que la planta 1 pierde eficiencia en la transformación (inversores).

In [ ]:
import pandas as pd

# Tabla de priorización de problemas
problemas = pd.DataFrame({
    'Problema': [
        'Baja eficiencia DC→AC (Planta 1)',
        'Generación DC baja (Planta 2)',
        'Calidad de datos inconsistente',
        'Conexiones intermitentes (Inversores P2)'
    ],
    'Planta': ['P1', 'P2', 'Ambas', 'P2'],
    'Impacto': ['Muy Alto', 'Alto', 'Medio', 'Alto'],
    'Prioridad': ['Crítica', 'Alta', 'Media', 'Alta'],
    'Acción': [
        'Revisar/recalibrar inversores DC-AC',
        'Inspeccionar módulos y conexiones',
        'Validar medidores y datalogger',
        'Revisar conexiones de inversores específicos'
    ]
})

print("\n" + "=" * 120)
print("MATRIZ DE PRIORIZACIÓN - PROBLEMAS OPERATIVOS")
print("=" * 120)
print(problemas.to_string(index=False))
print("=" * 120)

### Conclusiones Finales

**Situación actual:**
- Las dos plantas están generando por debajo de su capacidad teórica.
- La planta 1 tiene un problema claro en los inversores (10-15% eficiencia vs ~96% esperado).
- La planta 2 tiene un problema en generación DC (30-40% menos potencia disponible).

**Impacto económico estimado:**
- Ambas plantas pierden entre el 60-70% de su potencial de generación.
- Esto requiere intervención inmediata tanto en mantenimiento como en revisión de equipos.

**Próximos pasos:**
1. Validar que los datos de DC y AC están siendo medidos correctamente (especialmente en P1).
2. Realizar inspección física de los inversores en P1 y buscar signos de degradación o mala configuración.
3. En P2, revisar los módulos fotovoltaicos y sus conexiones en los inversores con baja generación.
4. Una vez identificada la raíz del problema, implementar plan de mantenimiento preventivo.

## Exportación de datos para Power BI

Con el objetivo de facilitar la construcción de dashboards y el seguimiento de indicadores operativos, exportamos el dataset final a formato CSV para su posterior explotación en Power BI.

In [ ]:
df_dia.reset_index().to_csv("kpi_fotovoltaica.csv", index=False)